# Animal Image Classifier — Training Notebook

This notebook handles data loading, model training, and evaluation.

CIFAR-10 has 10 classes. The three we want are:
- Class 2 → bird
- Class 3 → cat
- Class 5 → dog

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image

print('All libraries imported successfully')

## Step 1 — Define transforms

Transforms are applied to each image when it is loaded. For training we add random variations to help the model generalise. For validation and test we keep things consistent.

In [ ]:
# transforms applied to training images
# the random changes help the model see more variety and not just memorise the training set
train_transforms = transforms.Compose([
    # randomly flip the image left to right - a bird facing left is still a bird
    transforms.RandomHorizontalFlip(),
    # add a small border then randomly crop back to 32x32
    # this shifts the image slightly so the model doesnt rely on position
    transforms.RandomCrop(32, padding=4),
    # convert the image to a tensor - pixel values become numbers between 0 and 1
    transforms.ToTensor(),
    # normalise so values are roughly centred around 0 instead of 0 to 1
    # the mean and std values here were pre-calculated for the CIFAR-10 dataset
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

# transforms for validation and test images
# no random changes here - we want fair and consistent results
val_test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4914, 0.4822, 0.4465],
        std=[0.2023, 0.1994, 0.2010]
    )
])

print('Transforms defined')

## Step 2 — Create a filtered dataset class

CIFAR-10 has 60,000 images across 10 classes. We need to filter it to just the 3 animal classes and relabel them as 0, 1, 2.

In [ ]:
# this class wraps CIFAR-10 and only gives back the animal images we want
class AnimalDataset(Dataset):
    def __init__(self, cifar_data, animal_classes, label_map, transform=None):
        self.images = cifar_data.data       # all images as numpy arrays, shape (N, 32, 32, 3)
        self.labels = cifar_data.targets    # all labels as a list of integers
        self.transform = transform
        self.label_map = label_map

        # go through every label and collect the index if it is one of our 3 animals
        self.indices = []
        for i in range(len(self.labels)):
            if self.labels[i] in animal_classes:
                self.indices.append(i)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        # find the actual position in the original CIFAR-10 data
        real_idx = self.indices[idx]

        image = self.images[real_idx]   # numpy array: (32, 32, 3)
        label = self.labels[real_idx]

        # torchvision transforms expect a PIL Image, not a numpy array
        image = Image.fromarray(image)

        if self.transform is not None:
            image = self.transform(image)

        # remap: bird(2)->0, cat(3)->1, dog(5)->2
        new_label = self.label_map[label]
        return image, new_label

print('AnimalDataset class defined')

## Step 3 — Download CIFAR-10 and build datasets

In [ ]:
# the CIFAR-10 label numbers for our three animals
animal_classes = [2, 3, 5]
class_names = ['bird', 'cat', 'dog']

# map from CIFAR-10 labels to our labels (0, 1, 2)
label_map = {2: 0, 3: 1, 5: 2}

print('Downloading CIFAR-10 dataset...')

# download the raw data without transforms
# we pass the transforms into AnimalDataset so we can control them per split
full_train_raw = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True
)

full_test_raw = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True
)

print('Download complete!')
print(f'Total CIFAR-10 training images: {len(full_train_raw)}')
print(f'Total CIFAR-10 test images:     {len(full_test_raw)}')

In [ ]:
# create two filtered versions of the training data
# one with augmentation (for training) and one plain (for validation)
# keeping them separate means validation scores are not affected by random flips
train_and_val_augmented = AnimalDataset(full_train_raw, animal_classes, label_map, transform=train_transforms)
train_and_val_plain     = AnimalDataset(full_train_raw, animal_classes, label_map, transform=val_test_transforms)
test_dataset            = AnimalDataset(full_test_raw,  animal_classes, label_map, transform=val_test_transforms)

print(f'Animal images available for train+val: {len(train_and_val_augmented)}')
print(f'Animal images in test set:             {len(test_dataset)}')

# 80% for training, 20% for validation
total      = len(train_and_val_augmented)
val_size   = int(0.2 * total)
train_size = total - val_size

print(f'Splitting into {train_size} training and {val_size} validation images...')

# use the same fixed seed for both splits so the same images land in each split
generator = torch.Generator().manual_seed(42)
train_dataset, _ = random_split(train_and_val_augmented, [train_size, val_size], generator=generator)

generator = torch.Generator().manual_seed(42)
_, val_dataset   = random_split(train_and_val_plain,     [train_size, val_size], generator=generator)

print('Split done!')

## Step 4 — Create DataLoaders

In [ ]:
# how many images to feed the model at once
batch_size = 64

# DataLoader handles batching and shuffling automatically
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True    # shuffle training data so each epoch sees images in a different order
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False   # no need to shuffle when we are just evaluating
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print('--- Dataset Summary ---')
print(f'Training images:   {len(train_dataset)}')
print(f'Validation images: {len(val_dataset)}')
print(f'Test images:       {len(test_dataset)}')
print(f'Batch size:        {batch_size}')
print(f'Training batches:  {len(train_loader)}')
print(f'Classes:           {class_names}')

## Step 5 — Preview some training images

Let's display a few images to confirm the data loaded correctly and the labels match what we see.

In [ ]:
# grab one batch from the training loader
sample_images, sample_labels = next(iter(train_loader))

# the mean and std we used to normalise - we need these to undo it for display
mean = np.array([0.4914, 0.4822, 0.4465])
std  = np.array([0.2023, 0.1994, 0.2010])

# show the first 8 images in a 2x4 grid
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for i in range(8):
    img = sample_images[i].numpy()   # shape: (3, 32, 32)

    # undo the normalisation so the image displays with its original colours
    # formula: original_value = (normalised_value * std) + mean
    for channel in range(3):
        img[channel] = (img[channel] * std[channel]) + mean[channel]

    # rearrange from (3, 32, 32) to (32, 32, 3) because matplotlib expects channels last
    img = np.transpose(img, (1, 2, 0))

    # clip to 0-1 range just in case normalisation pushed any values outside
    img = np.clip(img, 0, 1)

    axes[i].imshow(img)
    axes[i].set_title(class_names[sample_labels[i].item()], fontsize=12)
    axes[i].axis('off')

plt.suptitle('Sample images from training set', fontsize=14)
plt.tight_layout()
plt.show()

print('Labels should match the animals in the images above')

## Step 6 — Load and test the CNN model

The model is defined in `model.py` in the project root. We import it here and do a quick test to confirm the output shape is correct.

In [ ]:
import sys
sys.path.append('..')  # add the project root so we can import model.py

from model import AnimalCNN

print('AnimalCNN imported successfully')

In [ ]:
# create an instance of the model
model = AnimalCNN()

# print the full architecture so we can see all the layers
print(model)

In [ ]:
# test with a fake batch to confirm the output shape is right
# input: 8 images, 3 colour channels, 32x32 pixels
dummy_input = torch.randn(8, 3, 32, 32)
output = model(dummy_input)

print(f'Input shape:  {dummy_input.shape}')
print(f'Output shape: {output.shape}')
print(f'Expected:     torch.Size([8, 3])  - 8 images, 3 class scores each')

# count how many trainable parameters the model has
total_params = 0
for param in model.parameters():
    total_params += param.numel()

print(f'Total trainable parameters: {total_params:,}')

---
## Model Evaluation

Run these cells after training is complete. We load the best saved model and test it on images it has never seen before.

> Make sure you have run all the cells above first so `test_loader` and `class_names` are defined.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# load the best model we saved during training
# map_location='cpu' means it will load even if it was trained on a GPU
model = AnimalCNN()
model.load_state_dict(torch.load('../model/best_model.pth', map_location='cpu'))
model.eval()

print('Model loaded successfully')

In [ ]:
# run the model on the entire test set and collect all predictions
correct_predictions = 0
total_images = 0

all_predictions = []
all_true_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)

        # get the class index with the highest score for each image
        _, predicted = torch.max(outputs, 1)

        correct_predictions += (predicted == labels).sum().item()
        total_images += labels.size(0)

        # save predictions and true labels so we can analyse them
        all_predictions.extend(predicted.numpy())
        all_true_labels.extend(labels.numpy())

test_accuracy = 100.0 * correct_predictions / total_images
print(f'Test Accuracy: {test_accuracy:.2f}%')

In [ ]:
# calculate how accurate the model was for each individual class
class_correct = [0, 0, 0]
class_total   = [0, 0, 0]

for true_label, predicted_label in zip(all_true_labels, all_predictions):
    class_total[true_label] += 1
    if true_label == predicted_label:
        class_correct[true_label] += 1

print(f'Test Accuracy: {test_accuracy:.2f}%')
print()
print('Per-class accuracy:')
for i, class_name in enumerate(class_names):
    class_acc = 100.0 * class_correct[i] / class_total[i]
    print(f'  {class_name}: {class_acc:.2f}%')

### Confusion Matrix

Each row is the **actual** class, each column is what the model **predicted**.
Numbers on the diagonal are correct predictions — everything off the diagonal is a mistake.

In [ ]:
# build and plot the confusion matrix
cm = confusion_matrix(all_true_labels, all_predictions)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title('Confusion Matrix — My CNN Model')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()

plt.savefig('../static/confusion_matrix.png')
print('Confusion matrix saved to static/confusion_matrix.png')
plt.show()

---
## Transfer Learning with ResNet18

Instead of building a CNN from scratch, we can take a model that was already trained on millions of images (ImageNet) and just retrain the last layer for our 3 classes.

This is called **transfer learning** — the model already knows how to detect edges, textures, and shapes. We just teach it to recognise our specific animals.

In [ ]:
from model import get_resnet18

# load pretrained resnet18 with the final layer swapped for our 3-class version
resnet_model = get_resnet18(num_classes=3)

# check how many parameters are actually being trained
# it should be just the final layer (512 * 3 weights + 3 biases = 1539)
trainable_params = 0
total_params = 0
for param in resnet_model.parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Frozen parameters:    {total_params - trainable_params:,}')
print()
print('Only the final layer is being trained - everything else is frozen')

In [ ]:
# training settings - same as before
resnet_num_epochs = 20
resnet_lr = 0.001
resnet_patience = 5

resnet_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
resnet_model = resnet_model.to(resnet_device)

# only pass the trainable parameters to the optimiser
resnet_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model.parameters()),
    lr=resnet_lr
)
resnet_criterion = nn.CrossEntropyLoss()
resnet_scheduler = torch.optim.lr_scheduler.StepLR(resnet_optimizer, step_size=7, gamma=0.1)

print('ResNet18 ready to train')

In [ ]:
# training loop for resnet18
resnet_train_losses     = []
resnet_val_losses       = []
resnet_train_accuracies = []
resnet_val_accuracies   = []

resnet_best_accuracy    = 0.0
resnet_early_stop_count = 0

print('Starting ResNet18 training...')
print('-' * 50)

for epoch in range(1, resnet_num_epochs + 1):

    # --- training pass ---
    resnet_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(resnet_device)
        labels = labels.to(resnet_device)

        resnet_optimizer.zero_grad()
        outputs = resnet_model(images)
        loss = resnet_criterion(outputs, labels)
        loss.backward()
        resnet_optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_acc  = 100.0 * correct / total

    # --- validation pass ---
    resnet_model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(resnet_device)
            labels = labels.to(resnet_device)

            outputs = resnet_model(images)
            loss = resnet_criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / len(val_loader)
    val_acc  = 100.0 * correct / total

    resnet_scheduler.step()

    resnet_train_losses.append(train_loss)
    resnet_val_losses.append(val_loss)
    resnet_train_accuracies.append(train_acc)
    resnet_val_accuracies.append(val_acc)

    print(f'Epoch {epoch}/{resnet_num_epochs}')
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
    print(f'  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%')

    if val_acc > resnet_best_accuracy:
        resnet_best_accuracy = val_acc
        torch.save(resnet_model.state_dict(), '../model/resnet18_best.pth')
        print('  Model improved! Saving...')
        resnet_early_stop_count = 0
    else:
        resnet_early_stop_count += 1
        print(f'  No improvement. ({resnet_early_stop_count}/{resnet_patience})')

    print('-' * 50)

    if resnet_early_stop_count >= resnet_patience:
        print(f'Early stopping triggered at epoch {epoch}')
        break

print(f'ResNet18 training complete! Best val accuracy: {resnet_best_accuracy:.2f}%')

In [ ]:
# evaluate resnet18 on the test set
resnet_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(resnet_device)
        labels = labels.to(resnet_device)

        outputs = resnet_model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

resnet_test_accuracy = 100.0 * correct / total
print(f'ResNet18 Test Accuracy: {resnet_test_accuracy:.2f}%')

In [ ]:
# compare the two models side by side
# test_accuracy was calculated earlier in the evaluation section above
print('Model Comparison:')
print(f'  My CNN Model:  {test_accuracy:.2f}%')
print(f'  ResNet18:      {resnet_test_accuracy:.2f}%')
print()

difference = resnet_test_accuracy - test_accuracy
if difference > 0:
    print(f'ResNet18 is {difference:.2f}% more accurate than my CNN')
else:
    print(f'My CNN is {abs(difference):.2f}% more accurate than ResNet18')

In [ ]:
# plot resnet18 training curves alongside my cnn curves for comparison
epochs_ran = len(resnet_train_losses)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# accuracy comparison
axes[0].plot(val_accuracies,         label='My CNN (val)',      linestyle='--')
axes[0].plot(resnet_val_accuracies,  label='ResNet18 (val)',    linestyle='-')
axes[0].set_title('Validation Accuracy Comparison')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()

# loss comparison
axes[1].plot(val_losses,             label='My CNN (val)',      linestyle='--')
axes[1].plot(resnet_val_losses,      label='ResNet18 (val)',    linestyle='-')
axes[1].set_title('Validation Loss Comparison')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig('../static/model_comparison.png')
print('Comparison plot saved to static/model_comparison.png')
plt.show()